# Proyecto de recuperación de la información
## Construcción del índice
A continuación descargamos la data que se encuentra en Huggingfaces con el corpus llamado fiqa

Instalamos la librería de datasets de Huggingface y descargamos el corpus fiqa

In [2]:
%pip install datasets
%pip install nltk


  Using cached datasets-5.2.0-py3-none-any.whl.metadata (24 kB)
  Using cached filelock-4.2.12-py3-none-any.whl.metadata (2.3 kB)
  Using cached pyarrow-26.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.1.0-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore2-2.16.2-py3-none-any.whl.metadata (28 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using c


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached nltk-3.10.4-py3-none-any.whl.metadata (5.0 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
Using cached nltk-3.10.4-py3-none-any.whl (1.7 MB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)

   ---------------------------------------- 0/5 [regex]
   ---------------------------------------- 0/5 [regex]
   -------- ------------------------------- 1/5 [defusedxml]
   -------- ------------------------------- 1/5 [defusedxml]
   -------- ------------------------------- 1/5 [defusedxml]
   -------- ------------------------------- 1/5 [defusedxml]
   ---------------- ----------------------- 2/5 [cloudpickle]
   ---------------- ----------------------- 2/5 [cloudpickle]
   ---------------- ----------------------- 2/5 [cloudpickle]
   ---------------- ----------------------- 2/5 [cloudpickle]
   ------------------------ ----------


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Importamos la librería datasets y cargamos el corpus fiqa, luego imprimimos el número de documentos, las columnas y el primer documento del corpus.

In [13]:
from datasets import load_dataset
corpus = load_dataset("BeIR/fiqa", "corpus", split = "corpus")
print("Número de documentos:", len(corpus))
print("Columnas", corpus.column_names)
documento = corpus[0]
print("ID\n", documento["_id"])
print("Título\n", documento["title"])
print("Texto\n", documento["text"])





Número de documentos: 57638
Columnas ['_id', 'title', 'text']
ID
 3
Título
 
Texto
 I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything.


Realizamos el procesamiento básico que incluye tokenización, normalización y remoción de stopwords. Para esto utilizamos la librería nltk y descargamos el conjunto de stopwords en español.

In [14]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
def tokenizar(texto):
    return re.findall(r"[a-z]+|[0-9]+(?:\.[0-9]+)?", texto.lower())
stopwords_ingles = set(stopwords.words("english"))
def quitar_stopwords(tokens):
    return[token for token in tokens if token not in stopwords_ingles]
tokens = tokenizar(corpus[0]["text"])
tokens_filtrados = quitar_stopwords(tokens)
print("Palabras en la lista de NLTK:", len(stopwords_ingles))
print("Antes:", tokens[:20])
print("Después:", tokens_filtrados[:20])



Palabras en la lista de NLTK: 198
Antes: ['i', 'm', 'not', 'saying', 'i', 'don', 't', 'like', 'the', 'idea', 'of', 'on', 'the', 'job', 'training', 'too', 'but', 'you', 'can', 't']
Después: ['saying', 'like', 'idea', 'job', 'training', 'expect', 'company', 'training', 'workers', 'job', 'building', 'software', 'perhaps', 'educational', 'systems', 'u', 'students', 'worry', 'little', 'getting']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
from collections import defaultdict
def construir_indice(documentos):
    indice = defaultdict(set)
    for doc in documentos:
        tokens = tokenizar(doc["text"])
        terminos = quitar_stopwords(tokens)
        for termino in set(terminos):
            indice[termino].add(doc["_id"])
    return indice
indice = construir_indice(corpus)
print("Indice construido:", len(indice))
print("Primeros 10 términos del índice:", list(indice.keys())[:10])


Indice construido: 74784
Primeros 10 términos del índice: ['getting', 'building', 'software', 'complaining', 'education', 'qualified', 'u', 'exchange', 'investment', 'thousands']


### Búsqueda basada en similitud Jaccard utilizando vectores binarios

In [16]:
from collections import Counter
tamanos = Counter()
for id_documentos in indice.values():
    tamanos.update(id_documentos)
def buscar_jaccard(consulta,indice,tamanos, k=10):
    terminos_consulta = set(quitar_stopwords(tokenizar(consulta)))
    coincidencias = Counter()
    for termino in terminos_consulta:
        coincidencias.update(indice.get(termino, ()))
    resultados = []
    for id_documento, interseccion in coincidencias.items():
        union = tamanos[id_documento] + len(terminos_consulta) - interseccion
        jaccard = interseccion / union
        resultados.append((id_documento, jaccard))
    resultados.sort(key=lambda resultado: (-resultado[1], resultado[0]))
    return resultados[:k]



In [17]:
documentos_prueba = [
    {"_id": "d1", "text": "credit"},
    {"_id": "d2", "text": "credit card loan bank"},
    {"_id": "d3", "text": "mortgage"},
]

indice_prueba = construir_indice(documentos_prueba)

tamanos_prueba = Counter()
for ids_documentos in indice_prueba.values():
    tamanos_prueba.update(ids_documentos)

resultados = buscar_jaccard("credit card", indice_prueba, tamanos_prueba)

# d1 obtiene 1/2 y d2 obtiene 2/4. d3 se omite porque no comparte términos.
assert dict(resultados) == {"d1": 0.5, "d2": 0.5}, resultados

resultados_repetidos = buscar_jaccard(
    "credit credit card", indice_prueba, tamanos_prueba
)
assert dict(resultados_repetidos) == dict(resultados), resultados_repetidos

print("Prueba de Jaccard aprobada")

Prueba de Jaccard aprobada


### TF-IDF

In [18]:
from math import log,sqrt
def construir_indice_tfidf(documentos):
    frecuencias = defaultdict(dict)

    for documento in documentos:
        doc_id = documento["_id"]
        tokens = quitar_stopwords(tokenizar(documento["text"]))

        for termino, frecuencia in Counter(tokens).items():
            frecuencias[termino][doc_id] = frecuencia

    n_documentos = len(documentos)
    idf = {}
    cuadrados = defaultdict(float)

    for termino, pesos_documentos in frecuencias.items():
        idf[termino] = log((n_documentos + 1) / (len(pesos_documentos) + 1)) + 1

        for doc_id, frecuencia in pesos_documentos.items():
            peso = frecuencia * idf[termino]
            pesos_documentos[doc_id] = peso
            cuadrados[doc_id] += peso ** 2

    normas = {doc_id: sqrt(total) for doc_id, total in cuadrados.items()}
    return frecuencias, idf, normas
def buscar_coseno(consulta, indice_tfidf, idf, normas, k=10):
    tokens = quitar_stopwords(tokenizar(consulta))
    pesos_consulta = {
        termino: frecuencia * idf[termino]
        for termino, frecuencia in Counter(tokens).items()
        if termino in idf
    }

    if not pesos_consulta:
        return []

    norma_consulta = sqrt(sum(peso ** 2 for peso in pesos_consulta.values()))
    productos = defaultdict(float)

    for termino, peso_consulta in pesos_consulta.items():
        for doc_id, peso_documento in indice_tfidf[termino].items():
            productos[doc_id] += peso_consulta * peso_documento

    resultados = [
        (doc_id, producto / (norma_consulta * normas[doc_id]))
        for doc_id, producto in productos.items()
    ]
    resultados.sort(key=lambda resultado: (-resultado[1], resultado[0]))
    return resultados[:k]

### BM25

In [12]:
from math import log1p


def construir_indice_bm25(documentos):
    indice_bm25 = defaultdict(dict)
    longitudes = {}

    for documento in documentos:
        doc_id = documento["_id"]
        tokens = quitar_stopwords(tokenizar(documento["text"]))
        longitudes[doc_id] = len(tokens)
        for termino, frecuencia in Counter(tokens).items():
            indice_bm25[termino][doc_id] = frecuencia

    n_documentos = len(longitudes)
    longitud_media = sum(longitudes.values()) / n_documentos if n_documentos else 0.0

    idf_bm25 = {}
    for termino, documentos_con_termino in indice_bm25.items():
        df = len(documentos_con_termino)
        idf_bm25[termino] = log1p((n_documentos - df + 0.5) / (df + 0.5))

    return indice_bm25, idf_bm25, longitudes, longitud_media


def buscar_bm25(consulta, indice_bm25, idf_bm25, longitudes, longitud_media,
                k=10, k1=1.2, b=0.75):
    if k <= 0 or longitud_media == 0:
        return []

    terminos_consulta = dict.fromkeys(quitar_stopwords(tokenizar(consulta)))
    puntajes = defaultdict(float)

    for termino in terminos_consulta:
        if termino not in indice_bm25:
            continue

        idf_termino = idf_bm25[termino]
        for doc_id, frecuencia in indice_bm25[termino].items():
            k_d = k1 * (1 - b + b * longitudes[doc_id] / longitud_media)
            aporte = idf_termino * frecuencia * (k1 + 1) / (frecuencia + k_d)
            puntajes[doc_id] += aporte

    resultados = sorted(puntajes.items(), key=lambda resultado: (-resultado[1], resultado[0]))
    return resultados[:k]


In [ ]:
# Ejemplo de las diapositivas 9 a 13.
documentos_bm25_clase = [
    {"_id": "D1", "text": "datos datos ciencia"},
    {"_id": "D2", "text": "datos ciencia modelos aprendizaje"},
    {"_id": "D3", "text": "ciencia modelos"},
]

datos_bm25_clase = construir_indice_bm25(documentos_bm25_clase)
_, idf_clase, longitudes_clase, longitud_media_clase = datos_bm25_clase
assert longitudes_clase == {"D1": 3, "D2": 4, "D3": 2}
assert longitud_media_clase == 3
assert round(idf_clase["datos"], 4) == 0.4700
assert round(idf_clase["ciencia"], 4) == 0.1335

ranking_clase = buscar_bm25("datos ciencia", *datos_bm25_clase)
assert [(doc_id, round(puntaje, 4)) for doc_id, puntaje in ranking_clase] == [
    ("D1", 0.7798),
    ("D2", 0.5311),
    ("D3", 0.1546),
]
assert buscar_bm25("datos ciencia datos", *datos_bm25_clase) == ranking_clase
assert buscar_bm25("hipoteca", *datos_bm25_clase) == []

for doc_id, puntaje in ranking_clase:
    print(f"{doc_id}: {puntaje:.4f}")


In [19]:
datos_bm25 = construir_indice_bm25(corpus)
consulta_bm25 = "teams"
for doc_id, puntaje in buscar_bm25(consulta_bm25, *datos_bm25, k=5):
    print(f"{doc_id}: {puntaje:.4f}")


401539: 11.4955
532293: 9.9128
223911: 9.8674
125371: 9.8223
303680: 9.7776


### Evaluación de los modelos

In [ ]:
from math import log2


def dcg(grados):
    return sum(
        (2 ** grado - 1) / log2(posicion + 1)
        for posicion, grado in enumerate(grados, start=1)
    )


def evaluar_ranking(ranking, juicios, cortes):
    total_relevantes = sum(grado >= 1 for grado in juicios.values())
    grados_ideales = sorted(juicios.values(), reverse=True)
    grados_ranking = []
    aciertos_acumulados = [0]
    aciertos = 0
    suma_precisiones = 0.0
    corte_maximo = max(cortes)

    # El ranking de estos modelos contiene un solo resultado por documento.
    for posicion, (doc_id, _) in enumerate(ranking, start=1):
        grado = juicios.get(str(doc_id), 0)
        if grado >= 1:
            aciertos += 1
            suma_precisiones += aciertos / posicion
        if posicion <= corte_maximo:
            grados_ranking.append(grado)
            aciertos_acumulados.append(aciertos)

    metricas = {"AP": suma_precisiones / total_relevantes if total_relevantes else 0.0}
    for k in cortes:
        aciertos_k = aciertos_acumulados[min(k, len(grados_ranking))]
        ideal_k = dcg(grados_ideales[:k])
        metricas[f"Precision@{k}"] = aciertos_k / k
        metricas[f"Recall@{k}"] = aciertos_k / total_relevantes if total_relevantes else 0.0
        metricas[f"NDCG@{k}"] = dcg(grados_ranking[:k]) / ideal_k if ideal_k else 0.0

    return metricas


In [ ]:
# Ranking A del ejemplo de las diapositivas 9–23.
juicios_ejemplo = {"d1": 3, "d2": 2, "d3": 2, "d4": 1}
ranking_ejemplo = [
    (doc_id, 0.0) for doc_id in ("d5", "d2", "d1", "d6", "d3", "d4", "d7", "d8")
]
metricas_ejemplo = evaluar_ranking(ranking_ejemplo, juicios_ejemplo, (5, 10))
assert metricas_ejemplo["Precision@5"] == 0.6
assert metricas_ejemplo["Recall@5"] == 0.75
assert round(metricas_ejemplo["NDCG@5"], 3) == 0.605
assert round(metricas_ejemplo["AP"], 4) == 0.6083
assert metricas_ejemplo["Precision@10"] == 0.4
assert all(valor == 0 for valor in evaluar_ranking([], {}, (5,)).values())


In [ ]:
consultas_fiqa = load_dataset("BeIR/fiqa", "queries", split="queries")
qrels_test = load_dataset("BeIR/fiqa-qrels", split="test")

texto_consultas = {str(fila["_id"]): fila["text"] for fila in consultas_fiqa}
juicios_test = defaultdict(dict)
for fila in qrels_test:
    query_id = str(fila["query-id"])
    doc_id = str(fila["corpus-id"])
    juicios_test[query_id][doc_id] = int(fila["score"])

ids_documentos = {str(doc_id) for doc_id in corpus["_id"]}
consultas_faltantes = juicios_test.keys() - texto_consultas.keys()
documentos_faltantes = {
    doc_id for juicios in juicios_test.values() for doc_id in juicios
} - ids_documentos
if not juicios_test or consultas_faltantes or documentos_faltantes:
    raise ValueError(
        f"Qrels de test incompatibles con FiQA: {len(juicios_test)} consultas evaluables, "
        f"{len(consultas_faltantes)} consultas y {len(documentos_faltantes)} documentos sin texto."
    )

print(f"Consultas de test: {len(juicios_test)} | Juicios: {len(qrels_test)}")


In [ ]:
from functools import partial
from statistics import mean

cortes = (5, 10, 20)
datos_tfidf = construir_indice_tfidf(corpus)
modelos = {
    "Jaccard": partial(buscar_jaccard, indice=indice, tamanos=tamanos),
    "TF-IDF + coseno": partial(buscar_coseno, indice_tfidf=datos_tfidf[0],
                               idf=datos_tfidf[1], normas=datos_tfidf[2]),
    "BM25": partial(buscar_bm25, indice_bm25=datos_bm25[0], idf_bm25=datos_bm25[1],
                    longitudes=datos_bm25[2], longitud_media=datos_bm25[3]),
}

# El límite del tamaño del corpus conserva todos los resultados para AP.
metricas_por_consulta = {}
resumen_evaluacion = {}
for nombre, buscar in modelos.items():
    detalle = {}
    for query_id, juicios in juicios_test.items():
        ranking = buscar(texto_consultas[query_id], k=len(corpus))
        detalle[query_id] = evaluar_ranking(ranking, juicios, cortes)
    metricas_por_consulta[nombre] = detalle

    resumen = {"MAP": mean(fila["AP"] for fila in detalle.values())}
    for k in cortes:
        for metrica in ("Precision", "Recall", "NDCG"):
            columna = f"{metrica}@{k}"
            resumen[columna] = mean(fila[columna] for fila in detalle.values())
    resumen_evaluacion[nombre] = resumen

# Mostrar una tabla Markdown sin añadir dependencias.
from IPython.display import Markdown, display

columnas = list(next(iter(resumen_evaluacion.values())))
filas = [
    "| Modelo | " + " | ".join(columnas) + " |",
    "| --- | " + " | ".join("---:" for _ in columnas) + " |",
]
for nombre, resumen in resumen_evaluacion.items():
    filas.append("| " + " | ".join([nombre] + [f"{resumen[c]:.4f}" for c in columnas]) + " |")
display(Markdown("\n".join(filas)))
